# 세번째 모델 생성

In [ ]:
from pathlib import Path
import shutil
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import cv2
import numpy as np
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO


# ============================================================
# 0. USER SETTINGS
# ============================================================

# Downloaded WTBD dataset folder path
DATA_ROOT = Path(r"C:\Users\user\OneDrive\바탕 화면\아이펠 자료\study\해커톤\wtbd\WT blade defect dataset\WT blade defect dataset")

# Ensemble model folder
# Put every .pt file you want to test here.
# After this path is set once, you only need to comment/uncomment file names below.
MODEL_DIR = Path(r"C:\Users\user\OneDrive\바탕 화면\아이펠 자료\study\해커톤")

# Ensemble model selection
# To enable/disable ensemble members, comment out ONLY the file-name line.
# Example:
# ENSEMBLE_MODEL_FILES = [
#     "yolov8s.pt",
#     # "yolov8s_augmented.pt",
# ]
ENSEMBLE_MODEL_FILES = [
    "yolov8s.pt",
    "yolo11s_norm750.pt",
    "yolo12s_norm750.pt",
    "yolov8s_norm1500.pt",
    "yolov10s_norm750.pt",
    "rtdetr-l.pt",
]

# Backward-compatible primary model path.
# The first uncommented model is used for class names and official Ultralytics val().
MODEL_PATH = MODEL_DIR / ENSEMBLE_MODEL_FILES[0]

# Ensemble NMS removes duplicate boxes predicted by multiple models.
ENSEMBLE_NMS_IOU_THRES = 0.55

# Output folder
OUT_ROOT = Path("verification_labeled_100_ensemble_recall_original_param_result")
TEST_DATA_ROOT = OUT_ROOT / "test_dataset"
PRED_IMAGE_DIR = OUT_ROOT / "predicted_images_merged_largebox"
RAW_PRED_IMAGE_DIR = OUT_ROOT / "predicted_images_raw"
COMPARE_IMAGE_DIR = OUT_ROOT / "gt_pred_comparison_images"
TOP_SAMPLE_DIR = OUT_ROOT / "top_detection_samples"
CONTAMINATION_IMAGE_DIR = OUT_ROOT / "contamination_candidate_images"
TOP_CONTAMINATION_SAMPLE_DIR = OUT_ROOT / "top_contamination_samples"
GRAPH_DIR = OUT_ROOT / "graphs"
REPORT_DIR = OUT_ROOT / "reports"

# Model input image size
IMG_SIZE = 1024

# Official validation threshold
# Keep this very low for mAP calculation.
VAL_CONF_THRES = 0.001

# Backtest / visualization thresholds
# ------------------------------------------------------------
# IMPORTANT PATCH FOR CONTAMINATION:
# If model.predict(conf=0.15) is used directly, low-confidence contamination/dirt
# candidates are discarded before our code can inspect or merge them.
#
# So we ask YOLO for low-threshold raw candidates first, then apply our own
# class-specific thresholds below.
#
# RAW_MODEL_CONF_THRES:
#   Used only when calling model.predict(). Keep low to avoid losing contamination.
#
# PRED_CONF_THRES:
#   Default threshold for ordinary classes.
#
# CONTAMINATION_CONF_THRES:
#   Lower threshold for dirt/contamination-like classes.
#
# DAMAGE_CONF_THRES:
#   Threshold for damage/defect-like classes.
RAW_MODEL_CONF_THRES = 0.03
PRED_CONF_THRES = 0.15
CONTAMINATION_CONF_THRES = 0.05
DAMAGE_CONF_THRES = 0.15
IOU_THRES = 0.5

# Original stricter visualization/backtest preset restored:
# - confidence thresholds are back to the earlier values
# - ensemble NMS is back to the earlier duplicate-removal setting
# - merged large boxes are enabled again for easier visual review

# ============================================================
# Large damage visualization/backtest patch
# ============================================================
# YOLO sometimes outputs several small boxes around one large damaged region.
# This option merges nearby damage boxes into one larger enclosing box for
# backtest-style visualization and custom TP/FP/FN counting.
# It does NOT retrain the model and it does NOT change official Ultralytics mAP.
USE_MERGED_LARGE_BOXES_FOR_BACKTEST = True
MERGE_TARGET_KEYWORDS = [
    # damage / defect
    "damage", "defect", "crack", "erosion", "scratch", "dent", "burn", "broken", "fault",
    # contamination / dirt
    "contamination", "contamin", "dirt", "dust", "pollution", "pollut", "stain", "soiling", "dirty"
]
# Boxes are considered connected if their expanded rectangles overlap.
# Increase this if one damaged area is still split into several boxes.
MERGE_GAP_RATIO = 0.08
# Add a small visual margin to the final merged box.
MERGED_BOX_PADDING_RATIO = 0.02
# Draw raw small boxes in a separate folder for debugging.
SAVE_RAW_PREDICTION_IMAGES = True

# None means use ALL images when RANDOM_LABELED_SAMPLE_ONLY is disabled.
MAX_IMAGES = None

# ============================================================
# Labeled 100-image recall verification mode
# ============================================================
# Keep labels and evaluate true object-level recall with IoU matching.
# The script randomly selects only images that actually contain at least one
# valid/mappable GT box, then compares ensemble predictions against those labels.
RANDOM_LABELED_SAMPLE_ONLY = True
LABELED_SAMPLE_SIZE = 100
LABELED_SAMPLE_SEED = 42
SELECTED_LABEL_MANIFEST_CSV = "selected_labeled_100_manifest.csv"

# Save all GT vs prediction comparison images?
# None = all. If CPU/storage is tight, set e.g. 100.
MAX_COMPARE_IMAGES = None

# Save top-N highest-confidence detection sample images separately.
TOP_N_SAMPLES = 30

# Whether to run Ultralytics official validation on the created test dataset.
RUN_OFFICIAL_VAL = False  # Primary-model only; ensemble recall is computed by custom IoU matching below.

# Whether XML classes may be merged into coarse model classes.
# Example: crack/erosion/scratch/dent -> damage
ALLOW_COARSE_MAPPING = True

DAMAGE_KEYWORDS = [
    "damage", "defect", "crack", "erosion", "scratch", "dent", "burn", "broken", "fault"
]

CONTAMINATION_KEYWORDS = [
    "contamination", "contamin", "dirt", "dirty", "dust", "pollution", "pollut", "stain", "soiling"
]

TARGET_KEYWORDS = DAMAGE_KEYWORDS + CONTAMINATION_KEYWORDS

IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


# ============================================================
# 1. SAFE IO UTILITIES
# ============================================================

def safe_imread(path: Path):
    """Read image safely on Windows paths with Korean/non-ASCII characters."""
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return img


def safe_imwrite(path: Path, image):
    """Save image safely on Windows paths with Korean/non-ASCII characters."""
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix if path.suffix else ".jpg"
    ok, encoded = cv2.imencode(ext, image)
    if not ok:
        raise RuntimeError(f"Image encoding failed: {path}")
    encoded.tofile(str(path))


def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


# ============================================================
# 2. FILE FINDING / LABEL PARSING
# ============================================================

def find_files(root: Path):
    images = []
    for ext in IMAGE_EXTS:
        images.extend(root.rglob(f"*{ext}"))
        images.extend(root.rglob(f"*{ext.upper()}"))

    xml_labels = list(root.rglob("*.xml"))
    txt_labels = list(root.rglob("*.txt"))
    yaml_files = list(root.rglob("data.yaml"))

    return sorted(set(images)), sorted(set(xml_labels)), sorted(set(txt_labels)), sorted(set(yaml_files))


def normalize_class_name(name: str) -> str:
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


def parse_voc_xml(xml_path: Path):
    """Read image size and objects from a PASCAL VOC XML file."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size_tag = root.find("size")
    if size_tag is None:
        return None, []

    img_w = int(float(size_tag.find("width").text))
    img_h = int(float(size_tag.find("height").text))

    objects = []
    for obj in root.findall("object"):
        name_tag = obj.find("name")
        box_tag = obj.find("bndbox")
        if name_tag is None or box_tag is None:
            continue

        class_name = name_tag.text.strip()
        xmin = float(box_tag.find("xmin").text)
        ymin = float(box_tag.find("ymin").text)
        xmax = float(box_tag.find("xmax").text)
        ymax = float(box_tag.find("ymax").text)

        objects.append({
            "class_name": class_name,
            "bbox": (xmin, ymin, xmax, ymax),
        })

    return (img_w, img_h), objects


def yolo_line_to_xyxy(line: str, img_w: int, img_h: int):
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    cls_id = int(float(parts[0]))
    xc, yc, bw, bh = map(float, parts[1:5])

    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h

    return cls_id, x1, y1, x2, y2


def box_iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


# ============================================================
# 3. CLASS MAPPING
# ============================================================

def find_model_class_by_keywords(model_names, keywords):
    for name in model_names:
        n = normalize_class_name(name)
        if any(k in n for k in keywords):
            return name
    return None


def build_xml_to_model_class_mapper(model_names):
    """
    Build mapper from source XML class names to trained model class names.

    This is necessary when WTBD has detailed defect labels but the trained model
    has coarse classes such as damage/dirt.
    """
    normalized_to_model = {normalize_class_name(name): name for name in model_names}

    dirt_keywords = [
        "dirt", "dust", "contamin", "contamination", "pollution",
        "pollut", "stain", "soiling"
    ]
    damage_keywords = [
        "damage", "damaged", "defect", "crack", "erosion", "scratch",
        "dent", "burn", "mark", "broken", "fault"
    ]

    dirt_model_name = find_model_class_by_keywords(model_names, dirt_keywords)
    damage_model_name = find_model_class_by_keywords(model_names, damage_keywords)

    print("\n" + "=" * 70)
    print("CLASS MAPPING CHECK")
    print("=" * 70)
    print("Model class used for dirt/contamination:", dirt_model_name)
    print("Model class used for damage/defect     :", damage_model_name)

    def mapper(xml_class_name: str):
        n = normalize_class_name(xml_class_name)

        # Exact class name match
        if n in normalized_to_model:
            return normalized_to_model[n]

        if not ALLOW_COARSE_MAPPING:
            return None

        # Dirt / contamination class
        if any(k in n for k in dirt_keywords) and dirt_model_name is not None:
            return dirt_model_name

        # Damage / defect class
        if any(k in n for k in damage_keywords) and damage_model_name is not None:
            return damage_model_name

        # WTBD is a defect dataset. If a damage-like class exists, map unknown defect classes to it.
        if damage_model_name is not None:
            return damage_model_name

        return None

    return mapper


def convert_voc_to_yolo(xml_path: Path, txt_path: Path, class_to_id: dict, class_mapper=None, skipped_counter=None):
    size, objects = parse_voc_xml(xml_path)
    if size is None:
        txt_path.write_text("", encoding="utf-8")
        return 0

    img_w, img_h = size
    lines = []

    for obj in objects:
        original_class_name = obj["class_name"]
        xmin, ymin, xmax, ymax = obj["bbox"]

        class_name = class_mapper(original_class_name) if class_mapper else original_class_name
        if class_name is None or class_name not in class_to_id:
            if skipped_counter is not None:
                skipped_counter[original_class_name] += 1
            continue

        class_id = class_to_id[class_name]

        xmin = max(0, min(xmin, img_w))
        xmax = max(0, min(xmax, img_w))
        ymin = max(0, min(ymin, img_h))
        ymax = max(0, min(ymax, img_h))

        box_w = xmax - xmin
        box_h = ymax - ymin
        if box_w <= 0 or box_h <= 0:
            continue

        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width = box_w / img_w
        height = box_h / img_h

        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    txt_path.write_text("\n".join(lines), encoding="utf-8")
    return len(lines)



def count_convertible_voc_boxes(xml_path: Path, class_to_id: dict, class_mapper=None):
    """Count XML boxes that will become valid YOLO boxes after class mapping."""
    size, objects = parse_voc_xml(xml_path)
    if size is None:
        return 0, Counter(), Counter()

    img_w, img_h = size
    class_counter = Counter()
    skipped_counter = Counter()
    box_count = 0

    for obj in objects:
        original_class_name = obj["class_name"]
        xmin, ymin, xmax, ymax = obj["bbox"]

        class_name = class_mapper(original_class_name) if class_mapper else original_class_name
        if class_name is None or class_name not in class_to_id:
            skipped_counter[original_class_name] += 1
            continue

        xmin = max(0, min(xmin, img_w))
        xmax = max(0, min(xmax, img_w))
        ymin = max(0, min(ymin, img_h))
        ymax = max(0, min(ymax, img_h))

        if xmax - xmin <= 0 or ymax - ymin <= 0:
            continue

        box_count += 1
        class_counter[class_name] += 1

    return box_count, class_counter, skipped_counter


def read_valid_yolo_rows_for_sampling(label_path: Path, num_classes: int):
    """Return valid YOLO rows, used only to find labeled TXT images for sampling."""
    if not label_path.exists():
        return []
    text = label_path.read_text(encoding="utf-8", errors="ignore").strip()
    if not text:
        return []

    rows = []
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
        except Exception:
            continue
        if 0 <= cls_id < num_classes and bw > 0 and bh > 0:
            rows.append((cls_id, xc, yc, bw, bh))
    return rows

def check_yolo_label_class_range(labels_dir: Path, num_classes: int):
    bad_rows = []
    total_boxes = 0

    for txt_path in labels_dir.rglob("*.txt"):
        text = txt_path.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            continue

        for line_no, line in enumerate(text.splitlines(), start=1):
            parts = line.strip().split()
            if len(parts) < 5:
                bad_rows.append((txt_path, line_no, line, "invalid_yolo_format"))
                continue

            try:
                cls_id = int(float(parts[0]))
            except ValueError:
                bad_rows.append((txt_path, line_no, line, "class_id_not_number"))
                continue

            total_boxes += 1
            if cls_id < 0 or cls_id >= num_classes:
                bad_rows.append((txt_path, line_no, line, f"class_id_out_of_range_for_nc_{num_classes}"))

    print("\n" + "=" * 70)
    print("YOLO LABEL RANGE CHECK")
    print("=" * 70)
    print("Label dir          :", labels_dir)
    print("Total boxes checked:", total_boxes)
    print("Bad rows           :", len(bad_rows))

    if bad_rows:
        print("First bad rows:")
        for row in bad_rows[:10]:
            print("-", row)
        raise RuntimeError("YOLO label class_id range does not match the trained model classes.")


def remove_ultralytics_cache(dataset_root: Path):
    cache_files = list(dataset_root.rglob("*.cache"))
    if cache_files:
        print("\nRemoving old Ultralytics cache files...")
    for p in cache_files:
        try:
            p.unlink()
            print("- removed", p)
        except Exception as e:
            print("- failed", p, e)


def find_ensemble_model_paths():
    """Resolve all uncommented ENSEMBLE_MODEL_FILES into existing model paths."""
    enabled_files = [str(name).strip() for name in ENSEMBLE_MODEL_FILES if str(name).strip()]
    if not enabled_files:
        raise RuntimeError(
            "No ensemble model files are enabled. Uncomment at least one file-name line in ENSEMBLE_MODEL_FILES."
        )

    resolved_paths = []
    for name in enabled_files:
        raw_path = Path(name).expanduser()
        candidates = []

        if raw_path.is_absolute():
            candidates.append(raw_path)
        else:
            candidates.extend([
                MODEL_DIR / name,
                Path.cwd() / name,
                Path(__file__).resolve().parent / name if "__file__" in globals() else Path.cwd() / name,
            ])

        found = None
        for candidate in candidates:
            if candidate.exists() and candidate.is_file():
                found = candidate.resolve()
                break

        if found is None:
            raise FileNotFoundError(
                f"Could not find ensemble model file: {name}\n"
                f"Checked MODEL_DIR and current/script folders. Current MODEL_DIR: {MODEL_DIR}"
            )

        resolved_paths.append(found)

    # Preserve order while removing accidental duplicates.
    unique_paths_out = []
    seen = set()
    for path in resolved_paths:
        key = str(path)
        if key not in seen:
            unique_paths_out.append(path)
            seen.add(key)

    return unique_paths_out


def load_ensemble_models():
    """Load all enabled YOLO models and verify that their class order matches."""
    model_paths = find_ensemble_model_paths()
    models = [YOLO(str(path)) for path in model_paths]
    model_labels = [path.name for path in model_paths]

    reference_names = [models[0].names[i] for i in sorted(models[0].names.keys())]
    for model_path, model in zip(model_paths[1:], models[1:]):
        names = [model.names[i] for i in sorted(model.names.keys())]
        if names != reference_names:
            raise RuntimeError(
                "Ensemble model class order mismatch. All models must have identical class IDs/names.\n"
                f"Reference ({model_paths[0].name}): {reference_names}\n"
                f"Mismatch  ({model_path.name}): {names}"
            )

    return models, model_paths, model_labels, reference_names


# ============================================================
# 4. CREATE TEST-ONLY DATASET, NO TRAIN/VAL SPLIT, NO LEARNING
# ============================================================

def make_test_dataset_from_xml(data_root: Path, out_root: Path, model_class_names, model_class_to_id):
    images, xml_labels, _, _ = find_files(data_root)

    image_map = {p.stem: p for p in images}
    xml_map = {p.stem: p for p in xml_labels}
    stems = sorted(set(image_map.keys()) & set(xml_map.keys()))
    all_pairs = [(image_map[s], xml_map[s]) for s in stems]

    if len(all_pairs) == 0:
        raise RuntimeError("No matched image-XML pairs found.")

    print("\nMatched image-XML pairs found:", len(all_pairs))

    xml_class_counter = Counter()
    for _, xml_path in all_pairs:
        _, objects = parse_voc_xml(xml_path)
        for obj in objects:
            xml_class_counter[obj["class_name"]] += 1

    print("\nClasses found in XML labels:")
    for name, count in xml_class_counter.most_common():
        print(f"- {name}: {count}")

    xml_to_model_class = build_xml_to_model_class_mapper(model_class_names)

    labeled_records = []
    skipped_counter_total = Counter()
    mapped_class_counter_total = Counter()

    for img_path, xml_path in all_pairs:
        mapped_box_count, mapped_class_counter, skipped_counter = count_convertible_voc_boxes(
            xml_path,
            model_class_to_id,
            class_mapper=xml_to_model_class,
        )
        skipped_counter_total.update(skipped_counter)
        mapped_class_counter_total.update(mapped_class_counter)
        if mapped_box_count > 0:
            labeled_records.append({
                "image_path": img_path,
                "label_path": xml_path,
                "mapped_gt_box_count": mapped_box_count,
                "mapped_class_names": ";".join(
                    f"{name}:{count}" for name, count in sorted(mapped_class_counter.items())
                ),
            })

    if not labeled_records:
        raise RuntimeError("No XML images with valid/mappable GT boxes were found.")

    if RANDOM_LABELED_SAMPLE_ONLY:
        sample_size = min(int(LABELED_SAMPLE_SIZE), len(labeled_records))
        rng = np.random.default_rng(LABELED_SAMPLE_SEED)
        sampled_indices = rng.choice(len(labeled_records), size=sample_size, replace=False)
        selected_records = [labeled_records[int(i)] for i in sampled_indices]
        selected_records = sorted(selected_records, key=lambda r: r["image_path"].name)
    else:
        selected_records = list(labeled_records)
        if MAX_IMAGES is not None:
            selected_records = selected_records[:MAX_IMAGES]

    print("\n" + "=" * 70)
    print("LABELED SAMPLE SELECTION")
    print("=" * 70)
    print("All matched image-XML pairs       :", len(all_pairs))
    print("Images with valid/mappable labels :", len(labeled_records))
    print("Selected labeled images           :", len(selected_records))
    print("Random labeled sample enabled     :", RANDOM_LABELED_SAMPLE_ONLY)
    if RANDOM_LABELED_SAMPLE_ONLY:
        print("Requested sample size             :", LABELED_SAMPLE_SIZE)
        print("Random seed                       :", LABELED_SAMPLE_SEED)

    if skipped_counter_total:
        print("\nSkipped XML classes that could not be mapped to model classes:")
        for name, count in skipped_counter_total.most_common():
            print(f"- {name}: {count}")

    if out_root.exists():
        shutil.rmtree(out_root)

    image_out = out_root / "images" / "test"
    label_out = out_root / "labels" / "test"
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    convert_skipped_counter = Counter()
    copied = 0
    total_boxes = 0
    manifest_rows = []

    for sample_rank, rec in enumerate(selected_records, start=1):
        img_path = Path(rec["image_path"])
        xml_path = Path(rec["label_path"])
        dst_img = image_out / img_path.name
        dst_txt = label_out / f"{img_path.stem}.txt"

        safe_copy(img_path, dst_img)
        box_count = convert_voc_to_yolo(
            xml_path,
            dst_txt,
            model_class_to_id,
            class_mapper=xml_to_model_class,
            skipped_counter=convert_skipped_counter,
        )
        copied += 1
        total_boxes += box_count

        manifest_rows.append({
            "sample_rank": sample_rank,
            "image": img_path.name,
            "source_image_path": str(img_path),
            "source_label_path": str(xml_path),
            "generated_image_path": str(dst_img),
            "generated_label_path": str(dst_txt),
            "mapped_gt_box_count": box_count,
            "mapped_class_names": rec.get("mapped_class_names", ""),
        })

    check_yolo_label_class_range(label_out, num_classes=len(model_class_names))

    data_yaml = {
        "path": str(out_root.resolve()),
        "train": "images/test",
        "val": "images/test",
        "test": "images/test",
        "names": model_class_names,
    }

    yaml_path = out_root / "data.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    pd.DataFrame(manifest_rows).to_csv(REPORT_DIR / SELECTED_LABEL_MANIFEST_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame([{
        "source_data_root": str(data_root),
        "selection_mode": "random_labeled_100" if RANDOM_LABELED_SAMPLE_ONLY else "all_labeled",
        "random_seed": LABELED_SAMPLE_SEED if RANDOM_LABELED_SAMPLE_ONLY else None,
        "requested_sample_size": LABELED_SAMPLE_SIZE if RANDOM_LABELED_SAMPLE_ONLY else None,
        "matched_image_label_pairs": len(all_pairs),
        "labeled_images_found": len(labeled_records),
        "images_selected": copied,
        "gt_boxes_written": total_boxes,
        "mapped_class_counter_all_labeled": dict(mapped_class_counter_total),
        "data_yaml": str(yaml_path),
    }]).to_csv(REPORT_DIR / "test_dataset_build_summary.csv", index=False, encoding="utf-8-sig")

    remove_ultralytics_cache(out_root)

    print("\n" + "=" * 70)
    print("TEST DATASET CREATED FROM 100 LABELED XML IMAGES")
    print("=" * 70)
    print("Images copied:", copied)
    print("Boxes written :", total_boxes)
    print("Manifest CSV  :", REPORT_DIR / SELECTED_LABEL_MANIFEST_CSV)
    print("data.yaml     :", yaml_path)

    return yaml_path

def make_test_dataset_from_yolo(data_root: Path, out_root: Path, model_class_names):
    images, _, txt_labels, yaml_files = find_files(data_root)
    image_map = {p.stem: p for p in images}
    label_map = {p.stem: p for p in txt_labels}
    stems = sorted(set(image_map.keys()) & set(label_map.keys()))
    all_pairs = [(image_map[s], label_map[s]) for s in stems]

    labeled_records = []
    class_counter_all_labeled = Counter()
    for img_path, txt_path in all_pairs:
        rows = read_valid_yolo_rows_for_sampling(txt_path, num_classes=len(model_class_names))
        if rows:
            class_counter_all_labeled.update([r[0] for r in rows])
            labeled_records.append({
                "image_path": img_path,
                "label_path": txt_path,
                "gt_box_count": len(rows),
                "class_ids": ";".join(str(r[0]) for r in rows),
            })

    if not labeled_records:
        raise RuntimeError("No matched image-TXT pairs with valid GT boxes found.")

    if RANDOM_LABELED_SAMPLE_ONLY:
        sample_size = min(int(LABELED_SAMPLE_SIZE), len(labeled_records))
        rng = np.random.default_rng(LABELED_SAMPLE_SEED)
        sampled_indices = rng.choice(len(labeled_records), size=sample_size, replace=False)
        selected_records = [labeled_records[int(i)] for i in sampled_indices]
        selected_records = sorted(selected_records, key=lambda r: r["image_path"].name)
    else:
        selected_records = list(labeled_records)
        if MAX_IMAGES is not None:
            selected_records = selected_records[:MAX_IMAGES]

    if out_root.exists():
        shutil.rmtree(out_root)

    image_out = out_root / "images" / "test"
    label_out = out_root / "labels" / "test"
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    manifest_rows = []
    total_boxes = 0
    for sample_rank, rec in enumerate(selected_records, start=1):
        img_path = Path(rec["image_path"])
        txt_path = Path(rec["label_path"])
        dst_img = image_out / img_path.name
        dst_txt = label_out / f"{img_path.stem}.txt"
        safe_copy(img_path, dst_img)
        safe_copy(txt_path, dst_txt)
        rows = read_valid_yolo_rows_for_sampling(dst_txt, num_classes=len(model_class_names))
        total_boxes += len(rows)
        manifest_rows.append({
            "sample_rank": sample_rank,
            "image": img_path.name,
            "source_image_path": str(img_path),
            "source_label_path": str(txt_path),
            "generated_image_path": str(dst_img),
            "generated_label_path": str(dst_txt),
            "gt_box_count": len(rows),
            "class_ids": ";".join(str(r[0]) for r in rows),
        })

    check_yolo_label_class_range(label_out, num_classes=len(model_class_names))

    data_yaml = {
        "path": str(out_root.resolve()),
        "train": "images/test",
        "val": "images/test",
        "test": "images/test",
        "names": model_class_names,
    }

    yaml_path = out_root / "data.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    pd.DataFrame(manifest_rows).to_csv(REPORT_DIR / SELECTED_LABEL_MANIFEST_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame([{
        "source_data_root": str(data_root),
        "selection_mode": "random_labeled_100" if RANDOM_LABELED_SAMPLE_ONLY else "all_labeled",
        "random_seed": LABELED_SAMPLE_SEED if RANDOM_LABELED_SAMPLE_ONLY else None,
        "requested_sample_size": LABELED_SAMPLE_SIZE if RANDOM_LABELED_SAMPLE_ONLY else None,
        "matched_image_label_pairs": len(all_pairs),
        "labeled_images_found": len(labeled_records),
        "images_selected": len(selected_records),
        "gt_boxes_written": total_boxes,
        "class_counter_all_labeled": dict(class_counter_all_labeled),
        "data_yaml": str(yaml_path),
    }]).to_csv(REPORT_DIR / "test_dataset_build_summary.csv", index=False, encoding="utf-8-sig")

    remove_ultralytics_cache(out_root)

    print("\n" + "=" * 70)
    print("TEST DATASET CREATED FROM 100 LABELED YOLO IMAGES")
    print("=" * 70)
    print("Matched image-TXT pairs           :", len(all_pairs))
    print("Images with valid labels found    :", len(labeled_records))
    print("Images selected for verification  :", len(selected_records))
    print("Boxes written                     :", total_boxes)
    print("Manifest CSV                      :", REPORT_DIR / SELECTED_LABEL_MANIFEST_CSV)
    print("data.yaml                         :", yaml_path)

    return yaml_path

# ============================================================
# 5. REPORT / DRAWING UTILITIES
# ============================================================

def count_yolo_labels(labels_dir: Path, class_names):
    counter = Counter()
    file_count = 0
    nonempty_file_count = 0
    total_boxes = 0
    areas = []

    for txt_path in sorted(labels_dir.glob("*.txt")):
        file_count += 1
        text = txt_path.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            continue
        nonempty_file_count += 1

        for line in text.splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(float(parts[0]))
            bw = float(parts[3])
            bh = float(parts[4])
            counter[cls_id] += 1
            total_boxes += 1
            areas.append(bw * bh)

    rows = []
    for cls_id, cls_name in enumerate(class_names):
        rows.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "gt_box_count": counter.get(cls_id, 0),
        })

    summary = {
        "label_files": file_count,
        "nonempty_label_files": nonempty_file_count,
        "total_boxes": total_boxes,
        "class_counter": dict(counter),
        "mean_box_area_ratio": sum(areas) / len(areas) if areas else 0.0,
    }

    return pd.DataFrame(rows), summary


def draw_ground_truth(image_bgr, label_path: Path, class_names):
    h, w = image_bgr.shape[:2]
    img = image_bgr.copy()

    if not label_path.exists():
        return img

    text = label_path.read_text(encoding="utf-8", errors="ignore").strip()
    for line in text.splitlines():
        parsed = yolo_line_to_xyxy(line, w, h)
        if parsed is None:
            continue
        cls_id, x1, y1, x2, y2 = parsed
        cls_name = class_names[cls_id] if 0 <= cls_id < len(class_names) else str(cls_id)
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(img, f"GT {cls_name}", (x1, max(25, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    return img


# ============================================================
# Large-box prediction post-processing utilities
# ============================================================

def is_merge_target_class(class_name: str) -> bool:
    """Return True if a class should use large-box merging."""
    n = normalize_class_name(class_name)
    return any(k in n for k in MERGE_TARGET_KEYWORDS)


def is_contamination_class(class_name: str) -> bool:
    """Return True for dirt/contamination-like model classes."""
    n = normalize_class_name(class_name)
    return any(k in n for k in CONTAMINATION_KEYWORDS)


def is_damage_class(class_name: str) -> bool:
    """Return True for damage/defect-like model classes."""
    n = normalize_class_name(class_name)
    return any(k in n for k in DAMAGE_KEYWORDS)


def get_class_threshold(class_name: str) -> float:
    """
    Class-specific threshold for backtest visualization.

    Contamination is often subtle and lower-contrast than cracks/broken regions,
    so using the same confidence threshold as damage can hide it completely.
    """
    if is_contamination_class(class_name):
        return CONTAMINATION_CONF_THRES
    if is_damage_class(class_name):
        return DAMAGE_CONF_THRES
    return PRED_CONF_THRES


def filter_pred_boxes_by_class_threshold(pred_boxes):
    """
    Apply class-specific confidence thresholds AFTER collecting low-threshold
    raw candidates from YOLO.
    """
    kept = []
    dropped_counter = Counter()

    for b in pred_boxes:
        th = get_class_threshold(b["class_name"])
        if float(b["confidence"]) >= th:
            b["class_threshold_used"] = th
            kept.append(b)
        else:
            dropped_counter[b["class_name"]] += 1

    return kept, dropped_counter


def clip_box_xyxy(box, img_w: int, img_h: int):
    x1, y1, x2, y2 = box
    x1 = max(0.0, min(float(x1), float(img_w)))
    x2 = max(0.0, min(float(x2), float(img_w)))
    y1 = max(0.0, min(float(y1), float(img_h)))
    y2 = max(0.0, min(float(y2), float(img_h)))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return x1, y1, x2, y2


def expand_box_xyxy(box, img_w: int, img_h: int, ratio: float):
    """Expand a box by a ratio of the image's longer side."""
    x1, y1, x2, y2 = box
    pad = max(img_w, img_h) * ratio
    return clip_box_xyxy((x1 - pad, y1 - pad, x2 + pad, y2 + pad), img_w, img_h)


def boxes_intersect(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return not (ax2 < bx1 or bx2 < ax1 or ay2 < by1 or by2 < ay1)


def union_box(boxes):
    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)
    return x1, y1, x2, y2


def get_prediction_boxes_from_result(result, class_names, source_model="model"):
    """Convert one Ultralytics result.boxes object to plain dictionaries."""
    pred_boxes = []
    if result.boxes is None or len(result.boxes) == 0:
        return pred_boxes

    for i in range(len(result.boxes)):
        cls_id = int(result.boxes.cls[i].item())
        conf = float(result.boxes.conf[i].item())
        x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
        class_name = class_names[cls_id] if 0 <= cls_id < len(class_names) else str(cls_id)
        pred_boxes.append({
            "class_id": cls_id,
            "class_name": class_name,
            "confidence": conf,
            "xmin": float(x1),
            "ymin": float(y1),
            "xmax": float(x2),
            "ymax": float(y2),
            "source": "raw_ensemble",
            "source_model": source_model,
            "source_models": source_model,
            "ensemble_support_count": 1,
            "merged_from_count": 1,
        })
    return pred_boxes


def nms_ensemble_boxes(pred_boxes, iou_thres=0.55):
    """
    Class-wise NMS for pooled ensemble boxes.

    Each enabled model predicts boxes. Duplicate boxes of the same class are
    collapsed, and the kept box records which model files supported that region.
    """
    if not pred_boxes:
        return []

    kept = []
    boxes_by_class = defaultdict(list)
    for b in pred_boxes:
        boxes_by_class[int(b["class_id"])].append(dict(b))

    for _, class_boxes in boxes_by_class.items():
        remaining = sorted(class_boxes, key=lambda x: float(x["confidence"]), reverse=True)
        while remaining:
            best = remaining.pop(0)
            best_box = (best["xmin"], best["ymin"], best["xmax"], best["ymax"])
            still_remaining = []
            supporting_models = set()
            if best.get("source_model"):
                supporting_models.add(str(best["source_model"]))

            for other in remaining:
                other_box = (other["xmin"], other["ymin"], other["xmax"], other["ymax"])
                if box_iou_xyxy(best_box, other_box) >= iou_thres:
                    if other.get("source_model"):
                        supporting_models.add(str(other["source_model"]))
                else:
                    still_remaining.append(other)

            best["source"] = "ensemble_nms"
            best["source_models"] = ",".join(sorted(supporting_models))
            best["ensemble_support_count"] = len(supporting_models)
            kept.append(best)
            remaining = still_remaining

    return sorted(kept, key=lambda x: float(x["confidence"]), reverse=True)


def predict_with_ensemble(models, model_labels, image_path: Path, class_names, imgsz: int, conf: float):
    """Run all enabled models on one image, pool boxes, then apply ensemble NMS."""
    pooled_boxes = []
    for model, model_label in zip(models, model_labels):
        result = model.predict(
            source=str(image_path),
            imgsz=imgsz,
            conf=conf,
            verbose=False,
        )[0]
        pooled_boxes.extend(get_prediction_boxes_from_result(result, class_names, source_model=model_label))
    return nms_ensemble_boxes(pooled_boxes, iou_thres=ENSEMBLE_NMS_IOU_THRES)


def merge_large_damage_boxes(pred_boxes, image_shape):
    """
    Merge nearby boxes for large damage visualization.

    Why this exists:
    - For blade damage, the true damaged region can be broad.
    - A detector may emit many small boxes around edges/fragments.
    - For backtest-style review, one larger enclosing box is often more readable.

    Important:
    - This cannot recover a large damaged area if the model produced no boxes near it.
    - Official mAP from model.val() still uses raw model predictions.
    """
    if not USE_MERGED_LARGE_BOXES_FOR_BACKTEST:
        return pred_boxes

    img_h, img_w = image_shape[:2]
    keep_boxes = []
    merge_candidates_by_class = defaultdict(list)

    for b in pred_boxes:
        if is_merge_target_class(b["class_name"]):
            merge_candidates_by_class[b["class_id"]].append(b)
        else:
            keep_boxes.append(b)

    merged_boxes = []

    for cls_id, boxes in merge_candidates_by_class.items():
        clusters = []

        for b in sorted(boxes, key=lambda x: x["confidence"], reverse=True):
            b_box = (b["xmin"], b["ymin"], b["xmax"], b["ymax"])
            b_expanded = expand_box_xyxy(b_box, img_w, img_h, MERGE_GAP_RATIO)

            matched_cluster_idx = None
            for ci, cluster in enumerate(clusters):
                cluster_box = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in cluster])
                cluster_expanded = expand_box_xyxy(cluster_box, img_w, img_h, MERGE_GAP_RATIO)
                if boxes_intersect(b_expanded, cluster_expanded):
                    matched_cluster_idx = ci
                    break

            if matched_cluster_idx is None:
                clusters.append([b])
            else:
                clusters[matched_cluster_idx].append(b)

        # Repeat once to merge clusters that became connected after union.
        changed = True
        while changed:
            changed = False
            new_clusters = []
            for cluster in clusters:
                c_box = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in cluster])
                c_expanded = expand_box_xyxy(c_box, img_w, img_h, MERGE_GAP_RATIO)

                merged_into_existing = False
                for existing in new_clusters:
                    e_box = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in existing])
                    e_expanded = expand_box_xyxy(e_box, img_w, img_h, MERGE_GAP_RATIO)
                    if boxes_intersect(c_expanded, e_expanded):
                        existing.extend(cluster)
                        changed = True
                        merged_into_existing = True
                        break

                if not merged_into_existing:
                    new_clusters.append(cluster)
            clusters = new_clusters

        for cluster in clusters:
            raw_union = union_box([(x["xmin"], x["ymin"], x["xmax"], x["ymax"]) for x in cluster])
            x1, y1, x2, y2 = expand_box_xyxy(raw_union, img_w, img_h, MERGED_BOX_PADDING_RATIO)
            best = max(cluster, key=lambda x: x["confidence"])
            mean_conf = sum(x["confidence"] for x in cluster) / len(cluster)

            cluster_source_models = set()
            for item in cluster:
                for model_name in str(item.get("source_models", item.get("source_model", ""))).split(","):
                    model_name = model_name.strip()
                    if model_name:
                        cluster_source_models.add(model_name)

            merged_boxes.append({
                "class_id": cls_id,
                "class_name": best["class_name"],
                "confidence": float(best["confidence"]),
                "mean_confidence": float(mean_conf),
                "xmin": x1,
                "ymin": y1,
                "xmax": x2,
                "ymax": y2,
                "source": "merged_largebox" if len(cluster) > 1 else "single_largebox",
                "source_models": ",".join(sorted(cluster_source_models)),
                "ensemble_support_count": len(cluster_source_models),
                "merged_from_count": len(cluster),
            })

    return keep_boxes + merged_boxes


def draw_prediction_boxes_from_list(image_bgr, pred_boxes, title_prefix="PRED"):
    """Draw prediction boxes from our post-processed box list."""
    img = image_bgr.copy()
    for b in pred_boxes:
        x1, y1, x2, y2 = map(int, [b["xmin"], b["ymin"], b["xmax"], b["ymax"]])
        cls_name = b["class_name"]
        conf = b["confidence"]
        merged_n = b.get("merged_from_count", 1)
        label = f"{title_prefix} {cls_name} {conf:.2f}"
        if merged_n > 1:
            label += f" x{merged_n}"
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 0), 3)
        cv2.putText(img, label, (x1, max(25, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    return img


def custom_match_at_iou_from_boxes(image_path: Path, label_path: Path, pred_boxes, class_names, iou_thres=0.5):
    """Custom TP/FP/FN using post-processed prediction boxes."""
    image = safe_imread(image_path)
    if image is None:
        return []

    h, w = image.shape[:2]
    gt_boxes = []

    text = label_path.read_text(encoding="utf-8", errors="ignore").strip() if label_path.exists() else ""
    for line in text.splitlines():
        parsed = yolo_line_to_xyxy(line, w, h)
        if parsed is None:
            continue
        cls_id, x1, y1, x2, y2 = parsed
        gt_boxes.append({"cls_id": cls_id, "box": (x1, y1, x2, y2), "matched": False})

    candidates = []
    for b in pred_boxes:
        candidates.append({
            "cls_id": int(b["class_id"]),
            "class_name": b["class_name"],
            "conf": float(b["confidence"]),
            "box": (b["xmin"], b["ymin"], b["xmax"], b["ymax"]),
            "source": b.get("source", "raw"),
            "merged_from_count": b.get("merged_from_count", 1),
        })

    candidates.sort(key=lambda x: x["conf"], reverse=True)

    rows = []
    for pred in candidates:
        best_iou = 0.0
        best_j = None
        for j, gt in enumerate(gt_boxes):
            if gt["matched"]:
                continue
            if gt["cls_id"] != pred["cls_id"]:
                continue
            iou = box_iou_xyxy(pred["box"], gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= iou_thres and best_j is not None:
            gt_boxes[best_j]["matched"] = True
            status = "TP"
        else:
            status = "FP"

        rows.append({
            "image": image_path.name,
            "class_id": pred["cls_id"],
            "class_name": pred["class_name"],
            "confidence": pred["conf"],
            "best_iou": best_iou,
            "status": status,
            "prediction_source": pred["source"],
            "merged_from_count": pred["merged_from_count"],
        })

    for gt in gt_boxes:
        if not gt["matched"]:
            rows.append({
                "image": image_path.name,
                "class_id": gt["cls_id"],
                "class_name": class_names[gt["cls_id"]] if 0 <= gt["cls_id"] < len(class_names) else str(gt["cls_id"]),
                "confidence": None,
                "best_iou": 0.0,
                "status": "FN",
                "prediction_source": "ground_truth_unmatched",
                "merged_from_count": 0,
            })

    return rows


def make_gt_pred_comparison(image_path: Path, label_path: Path, result, class_names, save_path: Path, pred_boxes=None):
    image = safe_imread(image_path)
    if image is None:
        return False

    gt_img = draw_ground_truth(image, label_path, class_names)

    if pred_boxes is None:
        pred_img = result.plot()
    else:
        pred_img = draw_prediction_boxes_from_list(image, pred_boxes, title_prefix="PRED")

    if pred_img.shape[:2] != gt_img.shape[:2]:
        pred_img = cv2.resize(pred_img, (gt_img.shape[1], gt_img.shape[0]))

    combined = cv2.hconcat([gt_img, pred_img])
    safe_imwrite(save_path, combined)
    return True


def custom_match_at_iou(image_path: Path, label_path: Path, result, class_names, iou_thres=0.5):
    image = safe_imread(image_path)
    if image is None:
        return []

    h, w = image.shape[:2]
    gt_boxes = []

    text = label_path.read_text(encoding="utf-8", errors="ignore").strip() if label_path.exists() else ""
    for line in text.splitlines():
        parsed = yolo_line_to_xyxy(line, w, h)
        if parsed is None:
            continue
        cls_id, x1, y1, x2, y2 = parsed
        gt_boxes.append({"cls_id": cls_id, "box": (x1, y1, x2, y2), "matched": False})

    pred_boxes = []
    if result.boxes is not None and len(result.boxes) > 0:
        for i in range(len(result.boxes)):
            cls_id = int(result.boxes.cls[i].item())
            conf = float(result.boxes.conf[i].item())
            x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
            pred_boxes.append({"cls_id": cls_id, "conf": conf, "box": (x1, y1, x2, y2)})

    pred_boxes.sort(key=lambda x: x["conf"], reverse=True)

    rows = []
    for pred in pred_boxes:
        best_iou = 0.0
        best_j = None
        for j, gt in enumerate(gt_boxes):
            if gt["matched"]:
                continue
            if gt["cls_id"] != pred["cls_id"]:
                continue
            iou = box_iou_xyxy(pred["box"], gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= iou_thres and best_j is not None:
            gt_boxes[best_j]["matched"] = True
            status = "TP"
        else:
            status = "FP"

        rows.append({
            "image": image_path.name,
            "class_id": pred["cls_id"],
            "class_name": class_names[pred["cls_id"]] if 0 <= pred["cls_id"] < len(class_names) else str(pred["cls_id"]),
            "confidence": pred["conf"],
            "best_iou": best_iou,
            "status": status,
        })

    for gt in gt_boxes:
        if not gt["matched"]:
            rows.append({
                "image": image_path.name,
                "class_id": gt["cls_id"],
                "class_name": class_names[gt["cls_id"]] if 0 <= gt["cls_id"] < len(class_names) else str(gt["cls_id"]),
                "confidence": None,
                "best_iou": 0.0,
                "status": "FN",
            })

    return rows


def make_top_sample_grid(det_df: pd.DataFrame, img_df: pd.DataFrame, save_path: Path, max_samples=12):
    if det_df.empty:
        return False

    top_images = (
        det_df.sort_values("confidence", ascending=False)
        .drop_duplicates("image")
        .head(max_samples)["saved_prediction_image"]
        .tolist()
    )

    thumbs = []
    for p in top_images:
        img = safe_imread(Path(p))
        if img is None:
            continue
        img = cv2.resize(img, (320, 240))
        thumbs.append(img)

    if not thumbs:
        return False

    cols = 3
    rows = int(np.ceil(len(thumbs) / cols))
    blank = np.zeros_like(thumbs[0])
    while len(thumbs) < rows * cols:
        thumbs.append(blank.copy())

    grid_rows = []
    for r in range(rows):
        grid_rows.append(cv2.hconcat(thumbs[r * cols:(r + 1) * cols]))
    grid = cv2.vconcat(grid_rows)

    safe_imwrite(save_path, grid)
    return True


def save_graphs(img_df: pd.DataFrame, det_df: pd.DataFrame, match_df: pd.DataFrame):
    GRAPH_DIR.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(14, 6))
    plt.plot(range(len(img_df)), img_df["max_confidence"])
    plt.title("Backtest detection probability by image")
    plt.xlabel("Image index")
    plt.ylabel("Max detection confidence")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.tight_layout()
    p = GRAPH_DIR / "overall_detection_probability_by_image.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print("Saved graph:", p)

    plt.figure(figsize=(10, 6))
    plt.hist(img_df["max_confidence"], bins=20)
    plt.title("Distribution of maximum detection confidence")
    plt.xlabel("Max confidence")
    plt.ylabel("Number of images")
    plt.xlim(0, 1)
    plt.grid(True)
    plt.tight_layout()
    p = GRAPH_DIR / "confidence_distribution.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print("Saved graph:", p)

    if not det_df.empty:
        class_count = det_df["class_name"].value_counts()
        plt.figure(figsize=(12, 6))
        class_count.plot(kind="bar")
        plt.title("Number of detections by class")
        plt.xlabel("Class")
        plt.ylabel("Detection count")
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y")
        plt.tight_layout()
        p = GRAPH_DIR / "detection_count_by_class.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved graph:", p)

        class_mean = det_df.groupby("class_name")["confidence"].mean().sort_values(ascending=False)
        plt.figure(figsize=(12, 6))
        class_mean.plot(kind="bar")
        plt.title("Average detection confidence by class")
        plt.xlabel("Class")
        plt.ylabel("Average confidence")
        plt.ylim(0, 1)
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y")
        plt.tight_layout()
        p = GRAPH_DIR / "average_confidence_by_class.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved graph:", p)

    if not det_df.empty and "is_contamination_like" in det_df.columns:
        contamination_df = det_df[det_df["is_contamination_like"] == True]
        if not contamination_df.empty:
            plt.figure(figsize=(10, 6))
            contamination_df["confidence"].hist(bins=20)
            plt.title("Contamination/dirt candidate confidence distribution")
            plt.xlabel("Confidence")
            plt.ylabel("Detection count")
            plt.xlim(0, 1)
            plt.grid(True)
            plt.tight_layout()
            p = GRAPH_DIR / "contamination_candidate_confidence_distribution.png"
            plt.savefig(p, dpi=200)
            plt.close()
            print("Saved graph:", p)

    if not match_df.empty:
        status_count = match_df["status"].value_counts()
        plt.figure(figsize=(8, 6))
        status_count.plot(kind="bar")
        plt.title(f"Backtest TP/FP/FN at conf={PRED_CONF_THRES}, IoU={IOU_THRES}")
        plt.xlabel("Status")
        plt.ylabel("Count")
        plt.grid(axis="y")
        plt.tight_layout()
        p = GRAPH_DIR / "custom_tp_fp_fn_count.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved graph:", p)


def save_recall_precision_reports(match_df: pd.DataFrame, img_df: pd.DataFrame, det_df: pd.DataFrame, class_names):
    """Save object-level and image-level recall reports from custom IoU matching."""
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    if match_df.empty:
        summary = {
            "tp": 0,
            "fp": 0,
            "fn": 0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "images_total": int(len(img_df)),
            "images_with_detection": int((img_df["detection_count"] > 0).sum()) if not img_df.empty else 0,
            "image_detection_rate": float((img_df["detection_count"] > 0).mean()) if not img_df.empty else 0.0,
        }
        pd.DataFrame([summary]).to_csv(REPORT_DIR / "recall_precision_summary.csv", index=False, encoding="utf-8-sig")
        return summary, pd.DataFrame()

    tp = int((match_df["status"] == "TP").sum())
    fp = int((match_df["status"] == "FP").sum())
    fn = int((match_df["status"] == "FN").sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    images_total = int(len(img_df))
    images_with_detection = int((img_df["detection_count"] > 0).sum()) if not img_df.empty else 0
    images_without_detection = int((img_df["detection_count"] == 0).sum()) if not img_df.empty else 0
    image_detection_rate = float(images_with_detection / images_total) if images_total else 0.0

    images_with_fn = int(match_df.loc[match_df["status"] == "FN", "image"].nunique())
    images_with_tp = int(match_df.loc[match_df["status"] == "TP", "image"].nunique())

    summary = {
        "evaluation_type": "ensemble_custom_iou_object_level",
        "iou_threshold": IOU_THRES,
        "raw_model_conf_threshold": RAW_MODEL_CONF_THRES,
        "default_prediction_conf_threshold": PRED_CONF_THRES,
        "damage_conf_threshold": DAMAGE_CONF_THRES,
        "contamination_conf_threshold": CONTAMINATION_CONF_THRES,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "images_total": images_total,
        "images_with_detection": images_with_detection,
        "images_without_detection": images_without_detection,
        "image_detection_rate": image_detection_rate,
        "images_with_tp": images_with_tp,
        "images_with_fn": images_with_fn,
    }
    pd.DataFrame([summary]).to_csv(REPORT_DIR / "recall_precision_summary.csv", index=False, encoding="utf-8-sig")

    class_rows = []
    for class_name in sorted(match_df["class_name"].dropna().unique()):
        cdf = match_df[match_df["class_name"] == class_name]
        ctp = int((cdf["status"] == "TP").sum())
        cfp = int((cdf["status"] == "FP").sum())
        cfn = int((cdf["status"] == "FN").sum())
        cprecision = ctp / (ctp + cfp) if (ctp + cfp) else 0.0
        crecall = ctp / (ctp + cfn) if (ctp + cfn) else 0.0
        cf1 = (2 * cprecision * crecall / (cprecision + crecall)) if (cprecision + crecall) else 0.0
        class_rows.append({
            "class_name": class_name,
            "tp": ctp,
            "fp": cfp,
            "fn": cfn,
            "precision": cprecision,
            "recall": crecall,
            "f1": cf1,
        })
    class_df = pd.DataFrame(class_rows)
    class_df.to_csv(REPORT_DIR / "recall_precision_by_class.csv", index=False, encoding="utf-8-sig")

    fn_df = match_df[match_df["status"] == "FN"].copy()
    fn_df.to_csv(REPORT_DIR / "false_negative_rows.csv", index=False, encoding="utf-8-sig")

    no_detection_df = img_df[img_df["detection_count"] == 0].copy() if not img_df.empty else pd.DataFrame()
    no_detection_df.to_csv(REPORT_DIR / "images_without_detection.csv", index=False, encoding="utf-8-sig")

    return summary, class_df


# ============================================================
# 6. MAIN BACKTEST / VERIFICATION
# ============================================================

def main():
    print("=" * 70)
    print("MODEL BACKTEST / TEST-ONLY VERIFICATION")
    print("=" * 70)
    print("This script does NOT train or fine-tune the model.")
    print("It creates a test-only dataset from 100 randomly selected labeled images, then prints/saves recall and GT-vs-prediction results.")

    for d in [OUT_ROOT, TEST_DATA_ROOT, PRED_IMAGE_DIR, RAW_PRED_IMAGE_DIR, COMPARE_IMAGE_DIR, TOP_SAMPLE_DIR, CONTAMINATION_IMAGE_DIR, TOP_CONTAMINATION_SAMPLE_DIR, GRAPH_DIR, REPORT_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    images, xml_labels, txt_labels, yaml_files = find_files(DATA_ROOT)

    print("\n" + "=" * 70)
    print("DATASET CHECK")
    print("=" * 70)
    print("DATA_ROOT :", DATA_ROOT)
    print("Images    :", len(images))
    print("XML labels:", len(xml_labels))
    print("TXT labels:", len(txt_labels))
    print("data.yaml :", len(yaml_files))

    if len(images) == 0:
        raise RuntimeError("No images found. Check DATA_ROOT.")

    print("\n" + "=" * 70)
    print("LOAD ENSEMBLE MODEL(S)")
    print("=" * 70)
    models, model_paths, model_labels, model_class_names = load_ensemble_models()
    model = models[0]  # Primary model. Used for class names and official Ultralytics val().
    MODEL_PATH = model_paths[0]
    model_class_to_id = {name: idx for idx, name in enumerate(model_class_names)}

    print("Primary model loaded:", MODEL_PATH)
    print("Ensemble models:")
    for i, path in enumerate(model_paths, start=1):
        print(f"- [{i}/{len(model_paths)}] {path}")
    print("Ensemble NMS IoU threshold:", ENSEMBLE_NMS_IOU_THRES)
    print("Model classes:")
    for idx, name in enumerate(model_class_names):
        print(f"- {idx}: {name}")

    contamination_model_classes = [name for name in model_class_names if is_contamination_class(name)]
    damage_model_classes = [name for name in model_class_names if is_damage_class(name)]

    print("\nDetected model class groups:")
    print("Damage-like classes       :", damage_model_classes)
    print("Contamination-like classes:", contamination_model_classes)

    if not contamination_model_classes:
        print("\nWARNING: No contamination/dirt-like class name was found in this model.")
        print("If your model was not trained with a contamination/dirt class, this code cannot create true contamination detections.")

    # Build test-only dataset from ALL available labeled images.
    if len(xml_labels) > 0:
        print("\nXML labels detected. Creating TEST-ONLY dataset from all XML labels...")
        data_yaml = make_test_dataset_from_xml(DATA_ROOT, TEST_DATA_ROOT, model_class_names, model_class_to_id)
    elif len(txt_labels) > 0:
        print("\nYOLO TXT labels detected. Creating TEST-ONLY dataset from all TXT labels...")
        data_yaml = make_test_dataset_from_yolo(DATA_ROOT, TEST_DATA_ROOT, model_class_names)
    else:
        data_yaml = None
        print("\nNo labels found. Official validation and TP/FP/FN matching will be skipped.")

    if data_yaml is not None:
        with open(data_yaml, "r", encoding="utf-8") as f:
            data_info = yaml.safe_load(f)
        dataset_path = Path(data_info["path"])
        test_image_dir = dataset_path / data_info["test"]
        test_label_dir = dataset_path / data_info["test"].replace("images", "labels")
        predict_images = sorted([p for p in test_image_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS])
    else:
        test_label_dir = None
        predict_images = images[:MAX_IMAGES] if MAX_IMAGES is not None else images

    print("\n" + "=" * 70)
    print("PREDICTION IMAGE CHECK")
    print("=" * 70)
    print("Prediction scope : RANDOM 100 LABELED IMAGES" if RANDOM_LABELED_SAMPLE_ONLY else "Prediction scope : ALL LABELED IMAGES")
    print("Prediction images:", len(predict_images))

    if data_yaml is not None:
        label_stats_df, label_summary = count_yolo_labels(test_label_dir, model_class_names)
        label_stats_csv = REPORT_DIR / "test_label_class_distribution.csv"
        label_stats_df.to_csv(label_stats_csv, index=False, encoding="utf-8-sig")

        print("\n" + "=" * 70)
        print("TEST LABEL DISTRIBUTION")
        print("=" * 70)
        print(label_stats_df.to_string(index=False))
        print("Label summary:", label_summary)
        print("Saved:", label_stats_csv)

        zero_gt = label_stats_df[label_stats_df["gt_box_count"] == 0]["class_name"].tolist()
        if zero_gt:
            print("\nWARNING: These model classes have zero GT boxes in this test dataset:", ", ".join(zero_gt))

        contamination_gt = label_stats_df[
            label_stats_df["class_name"].apply(is_contamination_class)
        ]["gt_box_count"].sum()
        if contamination_gt == 0:
            print("\nIMPORTANT: This test dataset has ZERO contamination/dirt ground-truth boxes.")
            print("Official mAP/recall cannot prove contamination performance on this dataset.")
            print("The script will still save contamination candidate predictions if the model outputs them.")

    # Official validation: no learning, only evaluation.
    metrics = None
    if data_yaml is not None and RUN_OFFICIAL_VAL:
        print("\n" + "=" * 70)
        print("OFFICIAL VALIDATION ON TEST-ONLY DATASET")
        print("=" * 70)
        print("No training will be run.")
        print("Official Ultralytics val() uses the PRIMARY model only:", MODEL_PATH)
        print("The backtest-style prediction loop below uses the ensemble.")
        print("Validation conf threshold:", VAL_CONF_THRES)
        print("Backtest/visualization conf threshold:", PRED_CONF_THRES)

        remove_ultralytics_cache(TEST_DATA_ROOT)
        check_yolo_label_class_range(test_label_dir, num_classes=len(model_class_names))

        metrics = model.val(
            data=str(data_yaml),
            split="test",
            imgsz=IMG_SIZE,
            conf=VAL_CONF_THRES,
            iou=IOU_THRES,
            plots=True,
            project=str(OUT_ROOT),
            name="official_validation",
            exist_ok=True,
        )

        print("\nOfficial validation result")
        print("mAP50-95:", metrics.box.map)
        print("mAP50   :", metrics.box.map50)
        print("mAP75   :", metrics.box.map75)

    # Backtest-style prediction loop
    print("\n" + "=" * 70)
    print("RUN BACKTEST-STYLE PREDICTION LOOP")
    print("=" * 70)

    detection_rows = []
    image_rows = []
    match_rows = []

    for idx, img_path in enumerate(predict_images, start=1):
        image_bgr = safe_imread(img_path)
        if image_bgr is None:
            print(f"Warning: could not read image: {img_path}")
            continue

        raw_pred_boxes = predict_with_ensemble(
            models=models,
            model_labels=model_labels,
            image_path=img_path,
            class_names=model_class_names,
            imgsz=IMG_SIZE,
            conf=RAW_MODEL_CONF_THRES,
        )

        # Apply class-specific thresholds after collecting low-confidence raw candidates.
        # This is especially important for contamination/dirt.
        filtered_pred_boxes, dropped_counter = filter_pred_boxes_by_class_threshold(raw_pred_boxes)

        used_pred_boxes = merge_large_damage_boxes(filtered_pred_boxes, image_bgr.shape)

        # Save raw ensemble visualization before class-specific thresholding.
        if SAVE_RAW_PREDICTION_IMAGES:
            raw_save_path = RAW_PRED_IMAGE_DIR / img_path.name
            raw_plotted = draw_prediction_boxes_from_list(image_bgr, raw_pred_boxes, title_prefix="RAW-ENS")
            safe_imwrite(raw_save_path, raw_plotted)
        else:
            raw_save_path = ""

        # Save backtest visualization using merged/large boxes.
        plotted = draw_prediction_boxes_from_list(image_bgr, used_pred_boxes, title_prefix="PRED")
        pred_save_path = PRED_IMAGE_DIR / img_path.name
        safe_imwrite(pred_save_path, plotted)

        has_contamination_candidate = any(is_contamination_class(b["class_name"]) for b in used_pred_boxes)
        if has_contamination_candidate:
            contamination_save_path = CONTAMINATION_IMAGE_DIR / img_path.name
            safe_imwrite(contamination_save_path, plotted)
        else:
            contamination_save_path = ""

        label_path = None
        compare_save_path = ""
        if test_label_dir is not None:
            label_path = test_label_dir / f"{img_path.stem}.txt"
            if MAX_COMPARE_IMAGES is None or idx <= MAX_COMPARE_IMAGES:
                compare_path = COMPARE_IMAGE_DIR / img_path.name
                make_gt_pred_comparison(img_path, label_path, None, model_class_names, compare_path, pred_boxes=used_pred_boxes)
                compare_save_path = str(compare_path)
            match_rows.extend(custom_match_at_iou_from_boxes(img_path, label_path, used_pred_boxes, model_class_names, iou_thres=IOU_THRES))

        max_conf = 0.0
        max_class_name = "none"
        detection_count = 0

        for b in used_pred_boxes:
            cls_id = int(b["class_id"])
            conf = float(b["confidence"])
            xmin, ymin, xmax, ymax = b["xmin"], b["ymin"], b["xmax"], b["ymax"]
            class_name = b["class_name"]
            is_damage_like = is_damage_class(class_name)
            is_contamination_like = is_contamination_class(class_name)
            is_target = is_damage_like or is_contamination_like

            detection_rows.append({
                "image": img_path.name,
                "image_path": str(img_path),
                "saved_prediction_image": str(pred_save_path),
                "saved_raw_prediction_image": str(raw_save_path),
                "saved_gt_pred_comparison_image": compare_save_path,
                "saved_contamination_candidate_image": str(contamination_save_path),
                "class_id": cls_id,
                "class_name": class_name,
                "confidence": conf,
                "mean_confidence": b.get("mean_confidence", conf),
                "class_threshold_used": b.get("class_threshold_used", get_class_threshold(class_name)),
                "prediction_source": b.get("source", "raw"),
                "source_models": b.get("source_models", b.get("source_model", "")),
                "ensemble_support_count": b.get("ensemble_support_count", 1),
                "ensemble_model_count": len(models),
                "merged_from_count": b.get("merged_from_count", 1),
                "is_damage_or_contamination_related": is_target,
                "is_damage_like": is_damage_like,
                "is_contamination_like": is_contamination_like,
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
            })

            detection_count += 1
            if conf > max_conf:
                max_conf = conf
                max_class_name = class_name

        image_rows.append({
            "image": img_path.name,
            "image_path": str(img_path),
            "saved_prediction_image": str(pred_save_path),
            "saved_raw_prediction_image": str(raw_save_path),
            "saved_gt_pred_comparison_image": compare_save_path,
            "saved_contamination_candidate_image": str(contamination_save_path),
            "has_contamination_candidate": has_contamination_candidate,
            "detection_count": detection_count,
            "max_confidence": max_conf,
            "top_class": max_class_name,
            "ensemble_model_count": len(models),
            "ensemble_models": ",".join(model_labels),
        })

        if idx % 50 == 0 or idx == len(predict_images):
            print(f"Processed {idx}/{len(predict_images)} images")

    det_df = pd.DataFrame(detection_rows)
    img_df = pd.DataFrame(image_rows)
    match_df = pd.DataFrame(match_rows)

    det_csv = OUT_ROOT / "detection_results.csv"
    img_csv = OUT_ROOT / "image_level_probability.csv"
    match_csv = OUT_ROOT / "custom_iou_match_summary.csv"

    det_df.to_csv(det_csv, index=False, encoding="utf-8-sig")
    img_df.to_csv(img_csv, index=False, encoding="utf-8-sig")
    match_df.to_csv(match_csv, index=False, encoding="utf-8-sig")

    contamination_df = det_df[det_df["is_contamination_like"] == True].copy() if not det_df.empty else pd.DataFrame()
    contamination_csv = OUT_ROOT / "contamination_candidate_results.csv"
    contamination_df.to_csv(contamination_csv, index=False, encoding="utf-8-sig")

    recall_summary, recall_by_class_df = save_recall_precision_reports(match_df, img_df, det_df, model_class_names)

    print("\nSaved CSV files:")
    print("-", det_csv)
    print("-", img_csv)
    print("-", match_csv)
    print("-", contamination_csv)
    print("-", REPORT_DIR / "recall_precision_summary.csv")
    print("-", REPORT_DIR / "recall_precision_by_class.csv")
    print("-", REPORT_DIR / "false_negative_rows.csv")
    print("-", REPORT_DIR / "images_without_detection.csv")

    print("\nENSEMBLE CUSTOM IoU RECALL SUMMARY")
    print("- TP       :", recall_summary.get("tp", 0))
    print("- FP       :", recall_summary.get("fp", 0))
    print("- FN       :", recall_summary.get("fn", 0))
    print("- Precision:", round(float(recall_summary.get("precision", 0.0)), 6))
    print("- Recall   :", round(float(recall_summary.get("recall", 0.0)), 6))
    print("- F1       :", round(float(recall_summary.get("f1", 0.0)), 6))
    print("- Image detection rate:", round(float(recall_summary.get("image_detection_rate", 0.0)), 6))

    # Save top detection samples
    if not det_df.empty:
        top_df = det_df.sort_values("confidence", ascending=False).head(TOP_N_SAMPLES)
        for rank, row in enumerate(top_df.itertuples(index=False), start=1):
            src = Path(row.saved_prediction_image)
            dst = TOP_SAMPLE_DIR / f"top_{rank:03d}_{row.confidence:.3f}_{row.class_name}_{row.image}"
            if src.exists():
                safe_copy(src, dst)

        grid_path = GRAPH_DIR / "top_detection_sample_grid.png"
        make_top_sample_grid(det_df, img_df, grid_path, max_samples=min(12, TOP_N_SAMPLES))
        print("Top sample images saved:", TOP_SAMPLE_DIR)
        print("Top sample grid saved  :", grid_path)

        # Save top contamination/dirt candidate samples separately.
        contamination_df = det_df[det_df["is_contamination_like"] == True].copy()
        if not contamination_df.empty:
            top_contam_df = contamination_df.sort_values("confidence", ascending=False).head(TOP_N_SAMPLES)
            for rank, row in enumerate(top_contam_df.itertuples(index=False), start=1):
                src = Path(row.saved_prediction_image)
                dst = TOP_CONTAMINATION_SAMPLE_DIR / f"contam_{rank:03d}_{row.confidence:.3f}_{row.class_name}_{row.image}"
                if src.exists():
                    safe_copy(src, dst)
            print("Top contamination samples saved:", TOP_CONTAMINATION_SAMPLE_DIR)
        else:
            print("No contamination/dirt candidates were found after class-specific thresholding.")

    save_graphs(img_df, det_df, match_df)

    # Text report
    report_path = REPORT_DIR / "backtest_summary.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("YOLO model backtest / test-only verification\n")
        f.write("=" * 70 + "\n\n")
        f.write("No training was performed in this script.\n")
        f.write(f"Primary model path: {MODEL_PATH}\n")
        f.write(f"Ensemble model count: {len(model_paths)}\n")
        f.write("Ensemble model paths:\n")
        for path in model_paths:
            f.write(f"- {path}\n")
        f.write(f"Ensemble NMS IoU threshold: {ENSEMBLE_NMS_IOU_THRES}\n")
        f.write(f"Data root : {DATA_ROOT}\n")
        f.write(f"Model classes: {model_class_names}\n")
        f.write(f"Total images used: {len(img_df)}\n")
        f.write(f"Random labeled sample only: {RANDOM_LABELED_SAMPLE_ONLY}\n")
        f.write(f"Labeled sample size: {LABELED_SAMPLE_SIZE}\n")
        f.write(f"Labeled sample seed: {LABELED_SAMPLE_SEED}\n")
        f.write(f"Selected label manifest: {REPORT_DIR / SELECTED_LABEL_MANIFEST_CSV}\n")
        f.write(f"RAW_MODEL_CONF_THRES: {RAW_MODEL_CONF_THRES}\n")
        f.write(f"PRED_CONF_THRES: {PRED_CONF_THRES}\n")
        f.write(f"CONTAMINATION_CONF_THRES: {CONTAMINATION_CONF_THRES}\n")
        f.write(f"DAMAGE_CONF_THRES: {DAMAGE_CONF_THRES}\n")
        f.write(f"IOU_THRES: {IOU_THRES}\n")
        f.write(f"USE_MERGED_LARGE_BOXES_FOR_BACKTEST: {USE_MERGED_LARGE_BOXES_FOR_BACKTEST}\n")
        f.write(f"MERGE_GAP_RATIO: {MERGE_GAP_RATIO}\n")
        f.write(f"MERGED_BOX_PADDING_RATIO: {MERGED_BOX_PADDING_RATIO}\n\n")

        if metrics is not None:
            f.write("Official validation\n")
            f.write("-" * 70 + "\n")
            f.write(f"mAP50-95: {metrics.box.map}\n")
            f.write(f"mAP50   : {metrics.box.map50}\n")
            f.write(f"mAP75   : {metrics.box.map75}\n\n")

        f.write("Ensemble custom IoU recall / precision\n")
        f.write("-" * 70 + "\n")
        f.write(f"tp: {recall_summary.get('tp', 0)}\n")
        f.write(f"fp: {recall_summary.get('fp', 0)}\n")
        f.write(f"fn: {recall_summary.get('fn', 0)}\n")
        f.write(f"precision: {float(recall_summary.get('precision', 0.0)):.6f}\n")
        f.write(f"recall: {float(recall_summary.get('recall', 0.0)):.6f}\n")
        f.write(f"f1: {float(recall_summary.get('f1', 0.0)):.6f}\n")
        f.write(f"image_detection_rate: {float(recall_summary.get('image_detection_rate', 0.0)):.6f}\n")
        f.write(f"recall_precision_summary_csv: {REPORT_DIR / 'recall_precision_summary.csv'}\n")
        f.write(f"recall_precision_by_class_csv: {REPORT_DIR / 'recall_precision_by_class.csv'}\n")
        f.write(f"false_negative_rows_csv: {REPORT_DIR / 'false_negative_rows.csv'}\n\n")

        f.write("Prediction summary\n")
        f.write("-" * 70 + "\n")
        f.write(f"Images with detection   : {int((img_df['detection_count'] > 0).sum())}\n")
        f.write(f"Images without detection: {int((img_df['detection_count'] == 0).sum())}\n")
        f.write(f"Average max confidence  : {float(img_df['max_confidence'].mean()):.6f}\n")
        f.write(f"Max confidence          : {float(img_df['max_confidence'].max()):.6f}\n\n")

        if not det_df.empty:
            f.write("Detected class counts\n")
            f.write("-" * 70 + "\n")
            for class_name, count in det_df["class_name"].value_counts().items():
                f.write(f"{class_name}: {count}\n")
            f.write("\n")

            if "is_contamination_like" in det_df.columns:
                contamination_count = int((det_df["is_contamination_like"] == True).sum())
                contamination_image_count = int(img_df.get("has_contamination_candidate", pd.Series(dtype=bool)).sum()) if "has_contamination_candidate" in img_df.columns else 0
                f.write("Contamination candidate summary\n")
                f.write("-" * 70 + "\n")
                f.write(f"contamination_candidate_boxes: {contamination_count}\n")
                f.write(f"images_with_contamination_candidate: {contamination_image_count}\n")
                if contamination_count == 0:
                    f.write("No contamination candidates were found. Check whether the model actually has a dirt/contamination class and whether it was trained for that class.\n")
                f.write("\n")

        if not match_df.empty:
            f.write("Custom backtest TP/FP/FN\n")
            f.write("-" * 70 + "\n")
            for status, count in match_df["status"].value_counts().items():
                f.write(f"{status}: {count}\n")
            tp = int((match_df["status"] == "TP").sum())
            fp = int((match_df["status"] == "FP").sum())
            fn = int((match_df["status"] == "FN").sum())
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f.write(f"custom_precision: {precision:.6f}\n")
            f.write(f"custom_recall   : {recall:.6f}\n")

    print("\n" + "=" * 70)
    print("FINAL BACKTEST SUMMARY")
    print("=" * 70)
    print("Total images used       :", len(img_df))
    print("Images with detection   :", int((img_df["detection_count"] > 0).sum()))
    print("Images without detection:", int((img_df["detection_count"] == 0).sum()))
    print("Average max confidence  :", round(float(img_df["max_confidence"].mean()), 4))
    print("Max confidence          :", round(float(img_df["max_confidence"].max()), 4))
    print("Ensemble model count    :", len(model_paths))
    print("Ensemble models         :", ", ".join(model_labels))
    print("Custom precision        :", round(float(recall_summary.get("precision", 0.0)), 6))
    print("Custom recall           :", round(float(recall_summary.get("recall", 0.0)), 6))
    print("Custom F1               :", round(float(recall_summary.get("f1", 0.0)), 6))

    if metrics is not None:
        print("\nOfficial validation:")
        print("mAP50-95:", metrics.box.map)
        print("mAP50   :", metrics.box.map50)
        print("mAP75   :", metrics.box.map75)

    if not det_df.empty:
        print("\nDetected classes:")
        for class_name, count in det_df["class_name"].value_counts().items():
            print(f"- {class_name}: {count}")

        if "is_contamination_like" in det_df.columns:
            contamination_count = int((det_df["is_contamination_like"] == True).sum())
            contamination_image_count = int(img_df["has_contamination_candidate"].sum()) if "has_contamination_candidate" in img_df.columns else 0
            print("\nContamination/dirt candidates:")
            print("- candidate boxes :", contamination_count)
            print("- candidate images:", contamination_image_count)

    if not match_df.empty:
        print("\nCustom backtest status:")
        for status, count in match_df["status"].value_counts().items():
            print(f"- {status}: {count}")

    print("\nSaved outputs:")
    print("- Test-only dataset        :", TEST_DATA_ROOT)
    print("- Predicted images merged  :", PRED_IMAGE_DIR)
    print("- Predicted images raw     :", RAW_PRED_IMAGE_DIR)
    print("- GT/Prediction comparison :", COMPARE_IMAGE_DIR)
    print("- Top detection samples    :", TOP_SAMPLE_DIR)
    print("- Contamination candidates :", CONTAMINATION_IMAGE_DIR)
    print("- Top contamination samples:", TOP_CONTAMINATION_SAMPLE_DIR)
    print("- Graphs                   :", GRAPH_DIR)
    print("- Reports                  :", REPORT_DIR)
    print("- Selected label manifest  :", REPORT_DIR / SELECTED_LABEL_MANIFEST_CSV)
    print("- Recall summary CSV       :", REPORT_DIR / "recall_precision_summary.csv")
    print("- Recall by class CSV      :", REPORT_DIR / "recall_precision_by_class.csv")
    print("- False negatives CSV      :", REPORT_DIR / "false_negative_rows.csv")
    print("- CSV results              :", OUT_ROOT)
    print("- Summary report           :", report_path)


if __name__ == "__main__":
    main()


# 세번째 모델 소개

정우 님이 50 에포크 학습 완료된 모델 yolov8 버전과 yolo12s 버전 두개를  

앙상블 모델로 탐지해본 결과 모델 서로간의 보완을 해줌으로써 탐지를 더 잘하게 되는  

결과를 얻었다  

제가 만든 yolov8s.pt 모델과 민욱님이 만드신 

rtdetr-l.pt  

yolo11s_norm750.pt  

yolo12s_norm750.pt  

yolov8s_norm1500.pt  

yolov10s_norm750.pt  

모델을 앙상블 하여 탐지를 실행했다  

로보 플로우의 wtbd 데이터셋의 라벨이 있는 데이터 100개를 무작위로 추출해 탐지 시켜보았다  

# 왼쪽은 사진의 라벨이며 오른쪽은 모델의 예측이다  

![](./img/result/181.jpg)

라벨과 비교해서 기존 데미지 위의 스크래치 까지 데미지로 인식한 모델의 탐지 결과이다  

![](./img/result/191.jpg)

기존의 데미지만 탐지한 원본 라벨과는 다르게 우리가 만든 모델은 사진 왼쪽의 오염까지 탐지한 결과이다  

![](./img/result/217.jpg)

원본 레벨보다 좀 더 디테일한 탐지를 하고 있다  

데미지 안의 아주 작은 오염까지도 탐지한 결과이다  

![](./img/result/472.jpg)

라벨의 왼쪽 데미지를 세세하게 탐지하지 못했고 가운데 가로지른 데미지도 탐지를 해내지 못했다  

![](./img/result/688.jpg)

사진의 오른쪽 위의 작은 데미지를 추가 탐지 했지만 큰 데미지의 박스 위치가 라벨과는 차이가 많이 난다  

![](./img/result/936.jpg)

모델이 불필요한 부분까지 박스를 그리고 있다  

# 드디어 우리가 원하는 상용화가 가능한 수준의 모델 까지 만들었다!!

처음에는 탐지 조차 못하는 경우가 많았는데  

학습 방법과을 바꾸고  

단일 모델에서 앙상블 모델로 바꾸니  

확실히 탐지 능력이 많이 올라갔다!!  


![](./img/result/chart.png)

# 어????

탐지 잘 된다고 모델 만들었다고 까불었는데  

실제 결과값은 참담했다  

recall 값이 14% 밖에 안되고  

mAP 50 - 95 값은 1%....  

상용화 할수 있는 수준이 아니다  